# Barranquilla MOPAC: QM/MM más rápido, validación y medida

El perfil anterior (`qmmm_prof_a100`, commit 89a30daf, tripsina–BPTI, 0,5 fs, tolerancia 0,01) dio 51,6 y 52,4 ms por
paso: MOPAC 32 ms, la parte Python de la fuerza 12,8 ms, largo alcance 3,3 ms, OpenMM y traspaso ~3 ms. Este commit
cambia cuatro cosas; cada una se puede apagar con una variable de entorno:

| Cambio | Variable para apagarlo |
|---|---|
| La parte Python sobre las ~46 000 cargas MM (imagen, grupos de conmutación, fuerzas) corre en la GPU con CuPy | `QMMM_MM_GPU=0` |
| `diagg2` con los coeficientes en memoria compartida (*staged*) por defecto en las sesiones QM/MM (−2 ms medidos) | `MOPAC_MOZYME_DIAGG2_STAGED=0` |
| Los búferes por llamada del gradiente, hcore, dispersión y puentes de H salen de un *pool* de CUDA (`cudaMallocAsync`) en vez de `cudaMalloc`/`cudaFree` (~100 por paso; `cudaFree` sincroniza la GPU) | `MOPAC_GPU_ASYNC_ALLOC=0` |
| Las tablas de parámetros del gradiente y de hcore se quedan en la GPU (antes: 24 reservas y 24 copias por llamada) | `MOPAC_GPU_PAIR_TABLE_CACHE=0` |
| El *scan* de la ordenación de `diagg2` corre en un bloque de 1024 hilos en vez de uno (~1 ms por paso) | (sin variable) |

| Bloque | Qué hace | Tiempo (A100) |
|---|---|---|
| Compilar | MOPAC con OpenBLAS | ~10 min |
| A | parte Python con MOPAC sustituido: fuerzas NumPy contra GPU en tres juegos de posiciones (incluidas moléculas partidas por la caja), y tiempos | ~2 min |
| B | tripsina, 400 pasos por variante desde el mismo estado: todo nuevo, todo apagado, cada cambio apagado por separado, repeticiones, tolerancia 0,001, perfil | ~9 min |
| B3 | regiones QM grandes (chignolina entera, crambina entera): nuevo, sin *staged*, todo apagado | ~5 min |
| C | `nsys` de 40 pasos de la variante nueva | ~3 min |

**Comprobación:** todas las corridas de una misma molécula parten del mismo estado, así que la primera llamada a MOPAC
es la misma geometría: su calor y sus fuerzas se comparan con la variante con todo apagado, y la diferencia entre dos
corridas iguales da la escala del ruido. En la tripsina, el calor de la primera llamada fue −297,0127265 kcal/mol en
el perfil anterior (mismo estado inicial), con diferencias de 2–6e-6 entre corridas.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Los
resultados se guardan en Google Drive (`MyDrive/mopac_paper_bench/RESULTS_NAME`); si la sesión se corta, vuelva a
ejecutar todo y lo terminado se salta. La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = '46637c1f'            # MM en la GPU, staged por defecto, pool de CUDA, tablas en la GPU, scan paralelo
RESULTS_NAME = 'qmmm_speed2_a100'
PREP_FROM = ['qmmm_dt_a100', 'qmmm_guess2_a100']   # estado inicial de la tripsina (el primero que exista)
NVT_STEPS = 400                # pasos de 0,5 fs por variante (tripsina)
USE_DRIVE = True
QUICK = False                  # True: pocos pasos (prueba del cuaderno)
REQUIRE_GPU = 'A100'           # None: aceptar cualquier GPU

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json, gzip
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno a {REQUIRE_GPU} '
                     f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/mopac_paper_bench')
else:
    BASE = Path('/content/mopac_paper_bench')
OUT = BASE / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC (biblioteca) con OpenBLAS

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available, fused_far_field_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available(), '; fused kernels:', fused_far_field_available())\"",
    check=False)

## 5. Herramientas y estados iniciales

- `/content/qmmm_prof.py`: tiempos por llamada a la fuerza QM/MM, calor, iteraciones SCF; fuerzas de la primera llamada
  (`QMMM_PROF_FORCES`).
- `/content/check_mm_gpu.py`: la parte Python con MOPAC sustituido, NumPy contra GPU.
- `/content/analyze_prof.py`: lectura de los perfiles y de `nsys`.

In [ ]:
Path('/content/qmmm_prof.py').write_text('"""Runs a QM/MM script unchanged (python qmmm_prof.py SCRIPT ARGS...) and records each call of the QM/MM force:\nstart time, duration of compute(), of MOPAC (QMSession.step) and of the long-range term, heat (kcal/mol),\nSCF iterations and MM charges seen.  QMMM_PROF_OUT: JSON file of these lists; QMMM_PROF_FORCES: .npz file of\nthe energy and forces of the first call.  QMMM_NSYS_RANGE=a,b: CUDA\nprofiler start before call a and stop after call b (nsys --capture-range=cudaProfilerApi)."""\nimport json\nimport os\nimport runpy\nimport sys\nimport time\n\nscript = sys.argv[1]\nsys.argv = sys.argv[1:]\nsys.path.insert(0, os.path.dirname(os.path.abspath(script)))\nimport openmm_mopac as om  # noqa: E402\nimport mopac_qmmm_api as api  # noqa: E402\n\nT = {k: [] for k in ("t0", "compute", "mopac", "pme_far", "pme_grad", "heat", "iters", "n_mm")}\nrng = os.environ.get("QMMM_NSYS_RANGE")\nrng = tuple(int(x) for x in rng.split(",")) if rng else None\nprof = None\nif rng:\n    try:\n        import cupy\n        prof = cupy.cuda.runtime\n    except Exception as e:  # noqa: BLE001\n        print("QMMM_NSYS_RANGE: no CuPy,", e, flush=True)\n\n\ndef wrap(cls, name, key):\n    f = getattr(cls, name)\n\n    def g(self, *a, **k):\n        t = time.perf_counter()\n        try:\n            return f(self, *a, **k)\n        finally:\n            T[key].append(time.perf_counter() - t)\n    setattr(cls, name, g)\n\n\nwrap(om.MopacQMMMForce, "_pme_far_potential", "pme_far")\nwrap(om.MopacQMMMForce, "_pme_far_gradient", "pme_grad")\nwrap(api.QMSession, "step", "mopac")\n_compute = om.MopacQMMMForce.compute\n\n\ndef compute(self, state):\n    k = len(T["t0"])\n    if prof is not None and k == rng[0]:\n        prof.deviceSynchronize()\n        prof.profilerStart()\n    t = time.perf_counter()\n    T["t0"].append(t)\n    r = _compute(self, state)\n    T["compute"].append(time.perf_counter() - t)\n    T["heat"].append(self.last.get("heat"))\n    T["iters"].append(self.session.scf_iterations)\n    T["n_mm"].append(self.last.get("n_mm"))\n    if k == 0 and os.environ.get("QMMM_PROF_FORCES"):\n        import numpy as np\n        np.savez(os.environ["QMMM_PROF_FORCES"], energy=r[0], forces=np.asarray(r[1]))\n    if prof is not None and k == rng[1]:\n        prof.deviceSynchronize()\n        prof.profilerStop()\n    return r\n\n\nom.MopacQMMMForce.compute = compute\ntry:\n    runpy.run_path(script, run_name="__main__")\nfinally:\n    out = os.environ.get("QMMM_PROF_OUT")\n    if out:\n        with open(out, "w") as fh:\n            json.dump(T, fh)\n')
Path('/content/check_mm_gpu.py').write_text('"""The QM/MM force with the MM-charge work in NumPy (QMMM_MM_GPU=0) and on the device (default with CuPy), MOPAC\nreplaced by an instant deterministic stub: forces on three position sets (as prepared; whole molecules shifted by\nbox vectors; single atoms shifted, molecules split) and time per call and per Verlet step.\n    python check_mm_gpu.py SCRIPTS_DIR PREP_DIR OUT_JSON [PLATFORM]"""\nimport json\nimport os\nimport sys\nimport time\n\nimport numpy as np\nimport openmm as mm\nimport openmm.app as app\nimport openmm.unit as u\n\nsys.path.insert(0, sys.argv[1])\nimport openmm_mopac as om  # noqa: E402\nfrom nve_drift import FF, make_system  # noqa: E402\nfrom run_qmmm_md import BACKBONE, parse_residues, residue_matches  # noqa: E402\n\nprep, out_json = sys.argv[2], sys.argv[3]\nplat = sys.argv[4] if len(sys.argv) > 4 else "CUDA"\n\n\nclass Stub:\n    def __init__(self, libpath, nums, x0, charge=0, model=0, tolerance=0.01, guess=None):\n        self.n = len(nums)\n        self.scf_iterations = 0\n\n    def step(self, xq, xs, qs, ron, roff, qm_pot=None):\n        xs = np.reshape(xs, (-1, 3))\n        d = xs - np.mean(xq, axis=0)\n        self.mm_pot = 0.1 * np.sin(7.0 * qs + 0.3 * d[:, 0])\n        self.qm_charges = 0.05 * np.cos(np.arange(self.n))\n        return float(np.sum(qs * self.mm_pot)), 0.02 * (xq - xq.mean(axis=0)), 0.01 * d\n\n    def close(self):\n        pass\n\n\nom.QMSession = Stub\nff = app.ForceField(*FF)\ntop = app.PDBFile(os.path.join(prep, "top.pdb")).topology\nwith open(os.path.join(prep, "state.xml")) as fh:\n    st0 = mm.XmlSerializer.deserialize(fh.read())\nqm = [x.index for x in top.atoms() for spec, sc in parse_residues("E:57:sc,E:102:sc,E:195:sc,I:15,I:16")\n      if residue_matches(x.residue, spec) and not (sc and x.name in BACKBONE)]\nx0 = np.array(st0.getPositions(asNumpy=True).value_in_unit(u.nanometer))\nL = np.diag(np.array(st0.getPeriodicBoxVectors(asNumpy=True).value_in_unit(u.nanometer)))\nrng = np.random.default_rng(5)\nxw = x0.copy()\nfor ch in top.chains():\n    if any(r.name in ("HOH", "NA", "CL") for r in ch.residues()):\n        for r in ch.residues():\n            xw[[a.index for a in r.atoms()]] += L * rng.integers(-1, 2, 3)\n    else:\n        xw[[a.index for a in ch.atoms()]] += L * rng.integers(-1, 2, 3)\nxsplit = xw.copy()\npick = rng.choice(len(x0), 300, replace=False)\nxsplit[pick] += L * rng.integers(-1, 2, (300, 3))\np = mm.Platform.getPlatformByName(plat)\nprops = {"Precision": "mixed"} if plat in ("CUDA", "OpenCL") else {}\nres, forces = {}, {}\nfor mode in ("0", ""):\n    os.environ["QMMM_MM_GPU"] = mode\n    system = make_system(ff, top, False)\n    qmmm = om.add_qmmm(system, top, st0.getPositions(), qm, 0, "stub", ron=10.0, roff=12.0)\n    name = "device" if qmmm.xp is not None and qmmm.xp is not np else "numpy"\n    integ = mm.VerletIntegrator(0.5 * u.femtoseconds)\n    ctx = mm.Context(system, integ, p, props)\n    ctx.setPeriodicBoxVectors(*st0.getPeriodicBoxVectors())\n    r = {"path": name, "far_class": None}\n    for tag, x in (("orig", x0), ("whole", xw), ("split", xsplit)):\n        ctx.setPositions(x * u.nanometer)\n        e, f = qmmm.compute(ctx.getState(getPositions=True))\n        forces[(mode, tag)] = (e, np.array(f))\n    r["far_class"] = type(qmmm.far).__name__\n    ctx.setPositions(st0.getPositions())\n    ctx.setVelocities(st0.getVelocities())\n    st = ctx.getState(getPositions=True)\n    for _ in range(5):\n        qmmm.compute(st)\n    t0 = time.perf_counter()\n    for _ in range(100):\n        qmmm.compute(st)\n    r["compute_ms"] = (time.perf_counter() - t0) / 100 * 1e3\n    integ.step(5)\n    t0 = time.perf_counter()\n    integ.step(200)\n    r["step_ms"] = (time.perf_counter() - t0) / 200 * 1e3\n    res[mode or "default"] = r\n    print(mode or "default", r, flush=True)\n    del ctx, integ\ncmp = {}\nfor tag in ("orig", "whole", "split"):\n    (e0, f0), (e1, f1) = forces[("0", tag)], forces[("", tag)]\n    cmp[tag] = dict(energy_diff=abs(e1 - e0), force_max_diff=float(np.abs(f1 - f0).max()),\n                    force_max=float(np.abs(f0).max()))\nres["compare"] = cmp\nprint(json.dumps(cmp, indent=1), flush=True)\nwith open(out_json, "w") as fh:\n    json.dump(res, fh, indent=1)\n')
Path('/content/analyze_prof.py').write_text('import re\nimport sqlite3\nfrom collections import defaultdict\n\nimport numpy as np\n\nRX_SEC = re.compile(r\'\\[PROFILE\\] MOZYME_SECTION name=(\\S+) calls=(\\d+) ms=([0-9.]+)\')\nRX_STAGE = re.compile(r\'\\[PROFILE\\] MOZYME_RESIDENT_STAGE name=(\\S+) calls=(\\d+) ms=([0-9.]+)\')\nRX_TRACE = re.compile(r\'\\[MOZYME GPU SCF trace\\] diagg2 staged=(\\d+) nij=(\\d+) rotated=(\\d+) max_chain=(\\d+) \'\n                      r\'lock_spins=(\\d+) nrej=(-?\\d+) lpt=(\\d+) grid=(\\d+) span_us=([0-9.]+) wait_us=([0-9.]+) \'\n                      r\'rotate_us=([0-9.]+)\')\n\n\ndef steady(T, skip=3):\n    """per-call medians (ms) after the first `skip` calls (the first SCF starts from the Lewis structure)"""\n    c, t0, m = np.array(T[\'compute\']), np.array(T[\'t0\']), np.array(T[\'mopac\'])\n    n = len(c)\n    far = (np.array(T[\'pme_far\']) + np.array(T[\'pme_grad\'])) if len(T[\'pme_far\']) == n else np.zeros(n)\n    s = slice(skip, n)\n    gap = t0[skip + 1:] - (t0[skip:-1] + c[skip:-1])\n    it = [x for x in T[\'iters\'][skip:] if x is not None]\n    r = dict(calls=n, step_ms=(t0[-1] + c[-1] - t0[skip]) / (n - skip) * 1e3,\n             compute_ms=np.median(c[s]) * 1e3, mopac_ms=np.median(m[s]) * 1e3, far_ms=np.median(far[s]) * 1e3,\n             python_ms=np.median(c[s] - m[s] - far[s]) * 1e3, gap_ms=np.median(gap) * 1e3,\n             gap_mean_ms=np.mean(gap) * 1e3, gaps_over_50ms=int(np.sum(gap > 0.05)),\n             scf_iterations=float(np.mean(it)) if it else None, mopac_ms_per_iteration=None,\n             first_call_ms=c[0] * 1e3, first_heat=T[\'heat\'][0], first_iters=T[\'iters\'][0])\n    if it:\n        r[\'mopac_ms_per_iteration\'] = r[\'mopac_ms\'] / np.mean(it)\n    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in r.items()}\n\n\ndef sections(text):\n    """MOPAC\'s cumulative section times (MOPAC_API_OUTPUT): the last value of each section"""\n    sec = {}\n    for m in RX_SEC.finditer(text):\n        sec[m.group(1)] = (int(m.group(2)), float(m.group(3)))\n    return sec\n\n\ndef stages(text):\n    """resident SCF stages (stdout), summed over the SCF calls"""\n    tot = defaultdict(lambda: [0, 0.0])\n    for m in RX_STAGE.finditer(text):\n        tot[m.group(1)][0] += int(m.group(2))\n        tot[m.group(1)][1] += float(m.group(3))\n    return dict(tot)\n\n\ndef trace(text):\n    """diagg2 sweeps (MOPAC_MOZYME_SCF_TRACE=1): medians over the sweeps"""\n    rows = [tuple(float(x) for x in m.groups()) for m in RX_TRACE.finditer(text)]\n    if not rows:\n        return {}\n    a = np.array(rows)\n    names = [\'staged\', \'nij\', \'rotated\', \'max_chain\', \'lock_spins\', \'nrej\', \'lpt\', \'grid\', \'span_us\', \'wait_us\',\n             \'rotate_us\']\n    r = {\'sweeps\': len(rows)}\n    r.update({n: float(np.median(a[:, i])) for i, n in enumerate(names)})\n    r[\'us_per_rotation_on_critical_path\'] = float(np.median(a[:, 8] / np.maximum(a[:, 3], 1)))\n    return r\n\n\ndef _union(iv):\n    iv = sorted(iv)\n    tot, cs, ce = 0, None, None\n    for s, e in iv:\n        if cs is None or s > ce:\n            if cs is not None:\n                tot += ce - cs\n            cs, ce = s, e\n        else:\n            ce = max(ce, e)\n    if cs is not None:\n        tot += ce - cs\n    return tot\n\n\ndef nsys_summary(sqlite_path, ncalls):\n    """GPU busy time against wall time over the captured range, launches and API calls per QM/MM call"""\n    con = sqlite3.connect(sqlite_path)\n    tables = {r[0] for r in con.execute("select name from sqlite_master where type=\'table\'")}\n    names = dict(con.execute(\'select id, value from StringIds\'))\n    k = con.execute(\'select start, end, shortName from CUPTI_ACTIVITY_KIND_KERNEL\').fetchall()\n    mc = con.execute(\'select start, end from CUPTI_ACTIVITY_KIND_MEMCPY\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_MEMCPY\' in tables else []\n    ms = con.execute(\'select start, end from CUPTI_ACTIVITY_KIND_MEMSET\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_MEMSET\' in tables else []\n    rt = con.execute(\'select start, end, nameId from CUPTI_ACTIVITY_KIND_RUNTIME\').fetchall() \\\n        if \'CUPTI_ACTIVITY_KIND_RUNTIME\' in tables else []\n    t_lo = min([r[0] for r in rt] + [r[0] for r in k])\n    t_hi = max([r[1] for r in rt] + [r[1] for r in k])\n    wall = (t_hi - t_lo) * 1e-6\n    busy = _union([(s, e) for s, e, _ in k] + mc + ms) * 1e-6\n    kern = defaultdict(lambda: [0, 0.0])\n    for s, e, nid in k:\n        kern[names.get(nid, str(nid))][0] += 1\n        kern[names.get(nid, str(nid))][1] += (e - s) * 1e-6\n    api = defaultdict(lambda: [0, 0.0])\n    for s, e, nid in rt:\n        api[names.get(nid, str(nid))][0] += 1\n        api[names.get(nid, str(nid))][1] += (e - s) * 1e-6\n    n = max(1, ncalls)\n    return dict(calls=ncalls, wall_ms_per_call=wall / n, gpu_busy_ms_per_call=busy / n,\n                kernel_ms_per_call=sum(v[1] for v in kern.values()) / n,\n                kernels_per_call=len(k) / n, memcpy_per_call=len(mc) / n, memset_per_call=len(ms) / n,\n                api_calls_per_call=len(rt) / n,\n                top_kernels=[(name, c / n, t / n) for name, (c, t) in sorted(kern.items(), key=lambda x: -x[1][1])[:30]],\n                top_api=[(name, c / n, t / n) for name, (c, t) in sorted(api.items(), key=lambda x: -x[1][1])[:20]])\n')

sys.path.insert(0, '/content')
import analyze_prof as AP
import urllib.request
import numpy as np
PROF = Path('/content/qmmm_prof.py')
QM = SRC / 'scripts' / 'qmmm'
ND = QM / 'nve_drift.py'
DATA = SRC / 'tests' / 'data' / 'mozyme_gpu_reference'
prep = OUT / 'runs' / 'prep'
if not (prep / 'state.xml').exists():
    prep.mkdir(parents=True, exist_ok=True)
    src = next((BASE / p / 'runs' / 'C_drift' / 'prep' for p in PREP_FROM
                if (BASE / p / 'runs' / 'C_drift' / 'prep' / 'state.xml').exists()), None)
    if src:
        for f in ('state.xml', 'top.pdb'):
            shutil.copy2(src / f, prep / f)
        (prep / 'from.txt').write_text(str(src) + '\n')
        print('estado inicial copiado de', src)
    else:
        pdb = OUT / 'runs' / '_pdb' / '2PTC.pdb'
        pdb.parent.mkdir(parents=True, exist_ok=True)
        if not pdb.exists():
            urllib.request.urlretrieve('https://files.rcsb.org/download/2PTC.pdb', pdb)
        print('no hay estado de', PREP_FROM, '; se prepara uno nuevo (50 ps de MM)')
        run(f'{sys.executable} -u {ND} prepare {pdb} --out {prep} --padding 1.0 --variants E:57=HID '
            f'--mm-equil-ps {10 if QUICK else 50} --platform CUDA --precision mixed', log=prep / 'log.txt')
N = 40 if QUICK else NVT_STEPS
QMSEL = '--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --ron 10 --roff 12 --precision mixed'
OLD = {'QMMM_MM_GPU': '0', 'MOPAC_GPU_ASYNC_ALLOC': '0', 'MOPAC_GPU_PAIR_TABLE_CACHE': '0',
       'MOPAC_MOZYME_DIAGG2_STAGED': '0'}
PROFILE = {'MOPAC_GPU_PROFILE': '1'}

def save_profiles(d, api_out, log_text, T):
    if T and T.get('compute'):
        (d / 'result.json').write_text(json.dumps(AP.steady(T), indent=1))
    sec = AP.sections(api_out.read_text(errors='ignore')) if api_out.exists() else {}
    if sec:
        with open(d / 'sections.csv', 'w') as fh:
            fh.write('section,calls,ms_total\n')
            for k, (c, ms) in sorted(sec.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    st = AP.stages(log_text)
    if st:
        with open(d / 'stages.csv', 'w') as fh:
            fh.write('stage,calls,ms_total\n')
            for k, (c, ms) in sorted(st.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    if api_out.exists():
        if sec:
            with open(api_out, 'rb') as fi, gzip.open(d / 'mopac_api.out.gz', 'wb') as fo:
                shutil.copyfileobj(fi, fo)
        api_out.unlink()

def nvt_run(group, name, prep_dir, qmsel, tol, envx, nvt):
    # one nve_drift.py run --nve 0 under qmmm_prof.py; returns (rc, result dict)
    d = OUT / 'runs' / group / name
    cmd = (f'{sys.executable} -u {PROF} {ND} run --prep {prep_dir} --out {d} --libmopac {LIBMOPAC} {qmsel} '
           f'--tolerance {tol} --platform CUDA --dt 0.5 --nvt {nvt} --nve 0 --seed 11')
    key = cmd + '\n' + json.dumps(envx, sort_keys=True)
    if (d / 'summary.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == key:
        print(group, name, 'ya hecho')
        return 0, json.loads((d / 'result.json').read_text()) if (d / 'result.json').exists() else {}
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    (d / 'cmd.txt').write_text(key)
    api_out = Path('/tmp') / f'mopac_api_{group}_{name}.out'
    api_out.unlink(missing_ok=True)
    env = dict(os.environ, OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_FORCEGPU='1',
               MOPAC_API_OUTPUT=str(api_out), QMMM_PROF_OUT=str(d / 'pytimes.json'),
               QMMM_PROF_FORCES=str(d / 'forces0.npz'), **envx)
    t0 = time.time()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    text = (d / 'log.txt').read_text(errors='ignore')
    T = json.loads((d / 'pytimes.json').read_text()) if (d / 'pytimes.json').exists() else None
    save_profiles(d, api_out, text, T)
    r = json.loads((d / 'result.json').read_text()) if (d / 'result.json').exists() else {}
    s = json.loads((d / 'summary.json').read_text()) if (d / 'summary.json').exists() else {}
    print(f"{group}/{name}: rc {rc}, {time.time() - t0:.0f} s; NVT {s.get('nvt_s_per_step', float('nan')) * 1e3:.1f} "
          f"ms/step; per call (median, ms): MOPAC {r.get('mopac_ms')}, far {r.get('far_ms')}, Python {r.get('python_ms')}, "
          f"gap {r.get('gap_ms')}; SCF iterations {r.get('scf_iterations')}; first heat {r.get('first_heat')}", flush=True)
    if rc != 0:
        print('    FALLÓ; final del registro:\n' + '\n'.join(text.strip().splitlines()[-15:]))
    return rc, r
print('prep:', (prep / 'from.txt').read_text().strip() if (prep / 'from.txt').exists() else prep)

## A. La parte Python: NumPy contra GPU (MOPAC sustituido)

Mismas entradas, fuerzas comparadas en tres juegos de posiciones: como están, moléculas enteras desplazadas por
vectores de la caja, y 300 átomos sueltos desplazados (moléculas partidas, como puede pasarle a OpenMM al envolver).

In [ ]:
dA = OUT / 'runs' / 'A_python'
dA.mkdir(parents=True, exist_ok=True)
if not (dA / 'check.json').exists():
    run(f'{sys.executable} -u /content/check_mm_gpu.py {QM} {prep} {dA}/check.json CUDA', check=False, log=dA / 'log.txt')
print((dA / 'check.json').read_text() if (dA / 'check.json').exists() else 'sin resultado (ver log.txt)')

## B. Tripsina–BPTI: variantes, 400 pasos NVT de 0,5 fs cada una

In [ ]:
VARIANTS = [('new', 0.01, {}), ('old_settings', 0.01, OLD), ('mm_host', 0.01, {'QMMM_MM_GPU': '0'}),
            ('no_async', 0.01, {'MOPAC_GPU_ASYNC_ALLOC': '0'}), ('no_table_cache', 0.01, {'MOPAC_GPU_PAIR_TABLE_CACHE': '0'}),
            ('no_staged', 0.01, {'MOPAC_MOZYME_DIAGG2_STAGED': '0'}), ('new_rep2', 0.01, {}), ('old_settings_rep2', 0.01, OLD),
            ('new_tol0.001', 0.001, {}), ('old_settings_tol0.001', 0.001, OLD), ('new_prof', 0.01, PROFILE)]
failed = []
for name, tol, envx in VARIANTS:
    rc, _ = nvt_run('B', name, prep, QMSEL, tol, envx, N)
    if rc:
        failed.append(name)
print('fallidas:', failed or 'ninguna')

## B3. Regiones QM grandes (*staged* pasa a ser el valor por defecto en toda sesión QM/MM)

Chignolina entera (138 átomos QM, caja de 2,5 nm de agua) y crambina entera (642 átomos QM, 3,0 nm), como en el
artículo (r_on/r_off 6/8 Å, PME), cada una preparada una vez (10 ps de MM) y corrida desde ese estado: nuevo, sin
*staged* y con todo apagado.

In [ ]:
BIG = [('chignolin', DATA / '1UAO_opt.pdb', 2.5, ','.join(map(str, range(1, 11))), 20 if QUICK else 200),
       ('crambin', DATA / '1CRN_opt.pdb', 3.0, ','.join(map(str, range(1, 47))), 10 if QUICK else 40)]
for mol, pdb, pad, res, nvt in BIG:
    pdir = OUT / 'runs' / 'B3' / f'prep_{mol}'
    if not (pdir / 'state.xml').exists():
        pdir.mkdir(parents=True, exist_ok=True)
        run(f'{sys.executable} -u {ND} prepare {pdb} --out {pdir} --padding {pad} --mm-equil-ps {2 if QUICK else 10} '
            f'--platform CUDA --precision mixed', check=False, log=pdir / 'log.txt')
    if not (pdir / 'state.xml').exists():
        print(mol, ': no se pudo preparar'); continue
    sel = f'--qm-residues {res} --ron 6 --roff 8 --precision mixed'
    for name, envx in (('new', {}), ('no_staged', {'MOPAC_MOZYME_DIAGG2_STAGED': '0'}), ('old_settings', OLD)):
        nvt_run('B3', f'{mol}_{name}', pdir, sel, 0.01, envx, nvt)

## C. `nsys` de la variante nueva (si está instalado)

Llamadas 20 a 59 (40 pasos). En el perfil anterior: GPU ocupada 26 de 54 ms por paso, 1703 kernels, 304 copias y 100
`cudaMalloc`/`cudaFree` por paso, y el *scan* de `diagg2` 1,06 ms por paso.

In [ ]:
import glob
NSYS = shutil.which('nsys') or next(iter(sorted(glob.glob('/usr/local/cuda*/bin/nsys') +
                                                glob.glob('/opt/nvidia/nsight-systems*/**/bin/nsys', recursive=True))), None)
print('nsys:', NSYS)
dC = OUT / 'runs' / 'C_nsys'
if NSYS and not (dC / 'nsys_summary.json').exists():
    shutil.rmtree(dC, ignore_errors=True); dC.mkdir(parents=True)
    env = dict(os.environ, OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_FORCEGPU='1',
               QMMM_PROF_OUT=str(dC / 'pytimes.json'), QMMM_NSYS_RANGE='20,59')
    cmd = (f'{NSYS} profile -t cuda,nvtx -s none --cpuctxsw=none --capture-range=cudaProfilerApi '
           f'--capture-range-end=stop -o {dC}/rep -f true {sys.executable} -u {PROF} {ND} run --prep {prep} '
           f'--out {dC}/run --libmopac {LIBMOPAC} {QMSEL} --tolerance 0.01 --platform CUDA --dt 0.5 --nvt 80 --nve 0 --seed 11')
    (dC / 'cmd.txt').write_text(cmd)
    with open(dC / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, cwd=dC, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    rep = next(dC.glob('rep*.nsys-rep'), None)
    print('rc', rc, 'report', rep)
    if rep:
        subprocess.run(f'{NSYS} stats -r cuda_api_sum,cuda_gpu_kern_sum,cuda_gpu_mem_time_sum -f csv -o {dC}/stats {rep}',
                       shell=True, stdout=open(dC / 'stats_log.txt', 'w'), stderr=subprocess.STDOUT)
        sq = next(dC.glob('rep*.sqlite'), None)
        try:
            s = AP.nsys_summary(str(sq), 40)
            (dC / 'nsys_summary.json').write_text(json.dumps(s, indent=1))
        except Exception as e:
            print('sqlite:', repr(e))
        if sq:
            sq.unlink()
if (dC / 'nsys_summary.json').exists():
    s = json.loads((dC / 'nsys_summary.json').read_text())
    print({k: v for k, v in s.items() if not k.startswith('top_')})
    for name, c, t in s['top_kernels'][:12]:
        print(f'  {t:8.3f} ms/paso  {c:7.1f} veces/paso  {name[:90]}')
    for name, c, t in s['top_api'][:12]:
        print(f'  {t:8.3f} ms/paso  {c:7.1f} veces/paso  {name[:90]}')

## D. Resultados

`speed2_runs.csv`: por corrida, tiempo por paso (NVT, de reloj), medianas por llamada (MOPAC, largo alcance, Python,
hueco), iteraciones SCF; y, contra la variante con todo apagado de la misma molécula (`old_settings`): diferencia del
calor y mayor diferencia de fuerza de la primera llamada (misma geometría), en kcal/mol y kJ/(mol nm).

In [ ]:
import pandas as pd
pd.set_option('display.width', 250)
rows = []
for group in ('B', 'B3'):
    gdir = OUT / 'runs' / group
    if not gdir.exists():
        continue
    for d in sorted(x for x in gdir.iterdir() if x.is_dir() and not x.name.startswith('prep')):
        r = json.loads((d / 'result.json').read_text()) if (d / 'result.json').exists() else {}
        s = json.loads((d / 'summary.json').read_text()) if (d / 'summary.json').exists() else {}
        mol = 'trypsin' if group == 'B' else d.name.split('_')[0]
        variant = d.name if group == 'B' else d.name.split('_', 1)[1]
        r.update(group=group, molecule=mol, variant=variant, nvt_ms_per_step=s.get('nvt_s_per_step', np.nan) * 1e3,
                 ok=(d / 'summary.json').exists())
        if (d / 'forces0.npz').exists():
            ref_name = ('old_settings_tol0.001' if variant.endswith('tol0.001') else 'old_settings')
            ref = gdir / (ref_name if group == 'B' else f'{mol}_old_settings') / 'forces0.npz'
            if ref.exists():
                a, b = np.load(d / 'forces0.npz'), np.load(ref)
                r['first_heat_diff_vs_old'] = float(a['energy'] - b['energy']) / 4.184
                r['first_force_maxdiff_vs_old'] = float(np.abs(a['forces'] - b['forces']).max())
                r['first_force_max'] = float(np.abs(b['forces']).max())
        rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(OUT / 'speed2_runs.csv', index=False)
display(df[[c for c in ['molecule', 'variant', 'ok', 'nvt_ms_per_step', 'mopac_ms', 'far_ms', 'python_ms', 'gap_ms',
                        'scf_iterations', 'first_heat', 'first_heat_diff_vs_old', 'first_force_maxdiff_vs_old',
                        'first_force_max'] if c in df]])
print('calor de la primera llamada, tripsina, contra el perfil anterior (-297.0127265):',
      {r['variant']: round(r['first_heat'] + 297.0127265, 7) for r in rows
       if r.get('molecule') == 'trypsin' and r.get('first_heat') and 'tol0.001' not in r['variant']})
st = OUT / 'runs' / 'B' / 'new_prof' / 'stages.csv'
if st.exists():
    x = pd.read_csv(st)
    calls = json.loads((OUT / 'runs' / 'B' / 'new_prof' / 'result.json').read_text())['calls']
    x['ms_per_step'] = x.ms_total / calls
    display(x.head(12))
for f in (OUT / 'runs' / 'A_python' / 'check.json',):
    print(f.parent.name, f.read_text() if f.exists() else 'sin resultado')

## E. Descargar

In [ ]:
tmp = Path('/content/_qmmm_speed2') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('speed2_runs.csv', 'env.json', 'commit.txt'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for pat in ('*.json', '*.csv', '*.txt', 'mopac_api.out.gz', 'forces0.npz'):
    for f in (OUT / 'runs').rglob(pat):
        if '_pdb' in f.parts or f.name == 'state.xml':
            continue
        dst = tmp / 'runs' / f.relative_to(OUT / 'runs')
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)